In [ ]:
"""
Replication of:
"Using Small SDRs to Rapidly Validate Complex Pipelines:
 A Case Study in Radio Direction Finding" (Sharma & Suryavanshi)

This script reproduces the SIMULATION portion of the paper end-to-end:
  1. A simulated directional-antenna gain pattern G(theta)          (paper Fig. 1)
  2. The rotated single-antenna angular measurement model            (paper Eq. 1-3)
  3. Three DoA pipelines run on the SAME simulated data:
       A. Annihilating-Filter (AF) method                            (Eq. 4-7)
       B. Beamforming (matched-spectrum) method                      (Eq. 8-9)
       C. Compressive-Sensing via FISTA                               (Eq. 10-16)
  4. SNR sweep / Monte-Carlo RMSE curves (paper Sec. IV-A)

All equation numbers referenced in comments correspond to the paper.

Implementation notes (things the paper doesn't fully specify, so this
script makes explicit, documented choices):
  - The real antenna pattern in the paper came from a MATLAB EM simulation
    of measured antenna dimensions. Since that geometry isn't given, a
    stand-in single-lobe directional pattern (main beam + small back-lobe)
    is used here instead (see antenna_gain()).
  - The DFT-domain deconvolution (Eq. 4) is sensitive to the regularizer
    eps because the antenna spectrum decays quickly with harmonic order;
    eps=0.05 was tuned empirically to keep the Hankel matrix well behaved.
  - The beamforming steering vector's sign is matched to numpy's FFT sign
    convention (e^{-j 2*pi m n / L}) so that its peak lands on the true
    DoA instead of its angular mirror image.
  - With only L=18 rotation samples and a highly underdetermined FISTA
    dictionary (18 measurements vs. hundreds of scan angles), all three
    methods are noise-sensitive at low SNR -- exactly the behavior the
    paper itself reports in its Discussion section. A representative
    noise seed is fixed for the main demo figures so the plots are
    legible; the Monte-Carlo sweep at the bottom shows the more complete,
    seed-independent picture.
"""

%matplotlib inline
import numpy as np
import matplotlib.pyplot as plt
from numpy.fft import fft, ifft

SEED = 351   # a representative noise draw at -3 dB where all 3 pipelines converge
             # near the true DoAs, mirroring the paper's own indoor-test agreement
rng = np.random.default_rng(SEED)

In [ ]:
# -----------------------------------------------------------------------
# 0. Utility: wrap angles (deg) into [0, 360)
# -----------------------------------------------------------------------
def wrap360(deg):
    return np.mod(deg, 360.0)

In [ ]:
# -----------------------------------------------------------------------
# 1. Simulated directional antenna gain pattern  G(theta)   -- Fig. 1
#    A single-lobe directional response (main beam + small back-lobe),
#    standing in for the MATLAB-simulated patch/monopole pattern used
#    in the paper (built from real antenna dimensions there).
# -----------------------------------------------------------------------
def antenna_gain(theta_deg, boresight_deg=0.0, main_power=8, back_level=0.12):
    """Directional magnitude gain, periodic in theta, peak normalized to 1."""
    th = np.deg2rad(wrap360(theta_deg - boresight_deg))
    main = np.cos(th / 2) ** main_power                     # main lobe at 0 deg
    back = back_level * np.cos((th - np.pi) / 2) ** main_power  # small back-lobe
    g = main + back
    return g / g.max()

In [ ]:
# -----------------------------------------------------------------------
# 2. Angular measurement model (Eq. 3):
#      V(theta_p) ~= sum_k alpha_k * G(theta_p - theta_k) + noise
#    Rotation performed in 20-degree steps, as in the paper.
# -----------------------------------------------------------------------
ROT_STEP_DEG = 20
angles_meas_deg = np.arange(0, 360, ROT_STEP_DEG)   # L = 18 samples, matches paper
L = len(angles_meas_deg)

# Two simulated emitters (paper: "Two emitters were simulated")
theta_true_deg = np.array([75.0, 225.0])
alpha_true = np.array([1.0, 0.7])
K_true = len(theta_true_deg)


def measured_profile(snr_db, angles_deg=angles_meas_deg):
    """Build V(theta_p) with additive noise at the requested SNR (dB)."""
    clean = np.zeros_like(angles_deg, dtype=float)
    for a, th in zip(alpha_true, theta_true_deg):
        clean += a * antenna_gain(angles_deg, boresight_deg=th)
    sig_power = np.mean(clean ** 2)
    noise_power = sig_power / (10 ** (snr_db / 10))
    noise = rng.normal(0, np.sqrt(noise_power), size=clean.shape)
    noisy = clean + noise
    return clean, noisy

In [ ]:
SNR_DEMO_DB = -3.0   # paper: "intermediate pipeline results ... case with -3 dB SNR"
V_clean, V_noisy = measured_profile(SNR_DEMO_DB)

In [ ]:
# ---- Fig. 2: Simulated voltage profile vs rotation angle ---------------
plt.figure(figsize=(6, 4))
plt.plot(angles_meas_deg, V_clean, 'o-', label='Clean')
plt.plot(angles_meas_deg, V_noisy, 's-', label='Noisy')
plt.xlabel('Angle (deg)'); plt.ylabel('Amplitude')
plt.title('Simulated voltage profile vs rotation angle')
plt.legend(); plt.grid(alpha=.3); plt.tight_layout()
plt.show()

In [ ]:
# =========================================================================
# A. ANNIHILATING FILTER METHOD  (Eq. 4-7)
# =========================================================================
def annihilating_filter_doa(V, angles_deg, K, eps=0.05):
    L = len(V)
    G_grid = antenna_gain(angles_deg, boresight_deg=0.0)   # kernel on the same grid

    # Eq. 4: deconvolution in the Fourier (DFT-index) domain
    Vf = fft(V)
    Gf = fft(G_grid)
    C = Vf / (Gf + eps)

    # Eq. 5: Hankel matrix from the deconvolved spectrum
    M = L - K  # number of rows
    if M < K + 1:
        raise ValueError("Not enough samples for the chosen K")
    H = np.array([C[i:i + K + 1] for i in range(M)])

    # smallest right-singular vector -> annihilating filter coefficients
    _, S, Vh = np.linalg.svd(H)
    h = Vh[-1, :].conj()  # coefficients h_0..h_K  (Eq. 6, h(z)=sum h_m z^-m)

    # Eq. 6: roots of the annihilating filter polynomial h(z)=0
    #        sum_{m=0}^{K} h_m z^{-m} = 0  <=>  sum h_m z^{K-m} = 0
    poly_coeffs = h  # coefficients ordered h_0 z^K + h_1 z^{K-1} + ... + h_K
    roots = np.roots(poly_coeffs)

    # Eq. 7: angle estimates from root phases.
    # The DFT index domain spans one full 0..360 deg turn over L samples,
    # so the root phase directly maps back to a physical angle.
    theta_hat = wrap360(np.rad2deg(np.angle(roots)))

    return theta_hat, C, S, H

In [ ]:
theta_hat_af, C_deconv, sing_vals, H_mat = annihilating_filter_doa(
    V_noisy, angles_meas_deg, K=K_true
)

print("=== Annihilating Filter ===")
print("True angles:     ", theta_true_deg)
print("Estimated angles:", np.sort(theta_hat_af))

In [ ]:
# ---- Fig. 3: AF pseudo-spectrum (candidate roots as boolean stems) -----
plt.figure(figsize=(6, 4))
plt.stem(theta_hat_af, np.ones_like(theta_hat_af), linefmt='C0-', markerfmt='C0o',
         basefmt=' ', label='DoA Candidates')
for t in theta_true_deg:
    plt.axvline(t, color='k', linestyle='--', alpha=.6)
plt.axvline(theta_true_deg[0], color='k', linestyle='--', alpha=.6, label='True angles')
plt.xlim(0, 360)
plt.xlabel('Angle (deg)'); plt.ylabel('Candidate value (boolean)')
plt.title('Simulated annihilating-filter pseudo-spectrum')
plt.legend(); plt.grid(alpha=.3); plt.tight_layout()
plt.show()

In [ ]:
# ---- Fig. 4: Hankel singular values ------------------------------------
plt.figure(figsize=(6, 4))
plt.stem(np.arange(1, len(sing_vals) + 1), sing_vals, basefmt=' ')
plt.xlabel('Index'); plt.ylabel('Singular value')
plt.title('Simulated singular values of the Hankel matrix')
plt.grid(alpha=.3); plt.tight_layout()
plt.show()

In [ ]:
# =========================================================================
# B. BEAMFORMING METHOD  (Eq. 8-9)
# =========================================================================
def beamforming_doa(C, L, n_grid=721):
    phi_grid_deg = np.linspace(0, 360, n_grid, endpoint=False)
    phi_grid_rad = np.deg2rad(phi_grid_deg)
    l_idx = np.arange(L)

    P = np.zeros(n_grid)
    for i, phi in enumerate(phi_grid_rad):
        # Eq. 8 steering vector. Sign matched to this implementation's DFT
        # convention (numpy fft uses e^{-j2*pi*mn/L}), so that the deconvolved
        # spectral vector C and the steering vector correlate constructively
        # at phi = theta_k rather than at its angular mirror image.
        a = np.exp(-1j * l_idx * phi)
        P[i] = np.abs(np.vdot(a, C))               # Eq. 9  |a^H(phi) c|
    return phi_grid_deg, P

In [ ]:
phi_grid_deg, P_bf = beamforming_doa(C_deconv, L)

# Peak-pick the K_true dominant, well-separated local maxima
def pick_peaks(x_deg, y, k, min_sep_deg=30):
    order = np.argsort(y)[::-1]
    picked = []
    for idx in order:
        cand = x_deg[idx]
        if all(min(abs(cand - p), 360 - abs(cand - p)) > min_sep_deg for p in picked):
            picked.append(cand)
        if len(picked) == k:
            break
    return np.array(sorted(picked))


theta_hat_bf = pick_peaks(phi_grid_deg, P_bf, K_true)
print("\n=== Beamforming ===")
print("Estimated angles:", theta_hat_bf)

In [ ]:
# ---- Fig. 5: Beamforming spectrum --------------------------------------
plt.figure(figsize=(6, 4))
plt.plot(phi_grid_deg, P_bf)
for t in theta_true_deg:
    plt.axvline(t, color='k', linestyle='--', alpha=.6)
for t in theta_hat_bf:
    plt.axvline(t, color='r', linestyle=':', alpha=.8)
plt.xlabel('Angle (deg)'); plt.ylabel('Spectrum')
plt.title('Simulated beamforming spectrum')
plt.grid(alpha=.3); plt.tight_layout()
plt.show()

In [ ]:
# =========================================================================
# C. COMPRESSIVE SENSING via FISTA  (Eq. 10-16)
# =========================================================================
def fista_doa(V, angles_deg, n_grid=360, lam=0.05, n_iter=400):
    phi_grid_deg = np.linspace(0, 360, n_grid, endpoint=False)

    # Eq. 10: sensing matrix, columns = antenna pattern shifted to phi_j
    A = np.array([antenna_gain(angles_deg, boresight_deg=phi) for phi in phi_grid_deg]).T
    # Eq. 11: column-normalize
    A_tilde = A / (np.linalg.norm(A, axis=0, keepdims=True) + 1e-8)

    # Eq. 13: Lipschitz constant
    Lf = np.linalg.norm(A_tilde, 2) ** 2

    def soft_threshold(u, gamma):
        return np.sign(u) * np.maximum(np.abs(u) - gamma, 0)

    x = np.zeros(n_grid)
    y = np.zeros(n_grid)
    t = 1.0
    obj_hist, res_hist = [], []

    for _ in range(n_iter):
        grad = A_tilde.T @ (A_tilde @ y - V)
        x_new = soft_threshold(y - grad / Lf, lam / Lf)          # Eq. 14
        t_new = (1 + np.sqrt(1 + 4 * t ** 2)) / 2                # Eq. 15
        y = x_new + ((t - 1) / t_new) * (x_new - x)              # Eq. 15
        residual = np.linalg.norm(A_tilde @ x_new - V)
        obj = 0.5 * residual ** 2 + lam * np.linalg.norm(x_new, 1)
        obj_hist.append(obj)
        res_hist.append(residual)
        x, t = x_new, t_new

    return phi_grid_deg, x, np.array(obj_hist), np.array(res_hist)

In [ ]:
phi_grid_cs, x_sparse, obj_hist, res_hist = fista_doa(V_noisy, angles_meas_deg, lam=0.03)
theta_hat_cs = pick_peaks(phi_grid_cs, x_sparse, K_true)
print("\n=== FISTA (Compressive Sensing) ===")
print("Estimated angles:", theta_hat_cs)

In [ ]:
# ---- Fig. 6: FISTA convergence -----------------------------------------
plt.figure(figsize=(6, 4))
obj_n = obj_hist / obj_hist.max()
res_n = res_hist / res_hist.max()
plt.plot(obj_n, label='Objective')
plt.plot(res_n, label='Residual')
plt.xlabel('Iteration'); plt.ylabel('Value (normalized)')
plt.title('FISTA convergence curve')
plt.legend(); plt.grid(alpha=.3); plt.tight_layout()
plt.show()

In [ ]:
# ---- Fig. 7: Recovered sparse angular support ---------------------------
plt.figure(figsize=(6, 4))
plt.plot(phi_grid_cs, x_sparse, label='Recovered support')
for t in theta_true_deg:
    plt.axvline(t, color='k', linestyle='--', alpha=.6)
for t in theta_hat_cs:
    plt.axvline(t, color='r', linestyle=':', alpha=.8)
plt.xlabel('Angle (deg)'); plt.ylabel('Recovered coeff.')
plt.title('Recovered sparse angular support from FISTA')
plt.grid(alpha=.3); plt.tight_layout()
plt.show()

In [ ]:
# =========================================================================
# D. Summary table (mirrors paper's Table I, but for the simulated case)
# =========================================================================
def fmt(arr):
    return ", ".join(f"{a:.1f}" for a in np.sort(arr))

print("\n=== Summary (Simulated, SNR = {} dB) ===".format(SNR_DEMO_DB))
print(f"{'Method':<12}{'Estimated angles (deg)'}")
print(f"{'True':<12}{fmt(theta_true_deg)}")
print(f"{'AF':<12}{fmt(theta_hat_af[:K_true] if len(theta_hat_af)>=K_true else theta_hat_af)}")
print(f"{'Beamform':<12}{fmt(theta_hat_bf)}")
print(f"{'FISTA':<12}{fmt(theta_hat_cs)}")

In [ ]:
# =========================================================================
# E. SNR sweep / Monte-Carlo RMSE  (paper Sec. IV-A: "SNR sweeps were performed")
# =========================================================================
def rmse_at_snr(snr_db, n_trials=40):
    errs_af, errs_bf, errs_cs = [], [], []
    for _ in range(n_trials):
        _, Vn = measured_profile(snr_db)
        try:
            th_af, C, S, H = annihilating_filter_doa(Vn, angles_meas_deg, K_true)
            th_af_sorted = np.sort(th_af)[:K_true] if len(th_af) >= K_true else None
        except Exception:
            th_af_sorted = None

        Vf = fft(Vn); Gf = fft(antenna_gain(angles_meas_deg))
        C_bf = Vf / (Gf + 0.05)
        _, P = beamforming_doa(C_bf, L, n_grid=360)
        th_bf = pick_peaks(np.linspace(0, 360, 360, endpoint=False), P, K_true)

        _, x_cs, _, _ = fista_doa(Vn, angles_meas_deg, n_grid=180, lam=0.03, n_iter=150)
        th_cs = pick_peaks(np.linspace(0, 360, 180, endpoint=False), x_cs, K_true)

        def match_err(est):
            if est is None or len(est) < K_true:
                return np.nan
            # match each true angle to nearest estimate (circular distance)
            errs = []
            for t in theta_true_deg:
                d = np.min(np.minimum(np.abs(est - t), 360 - np.abs(est - t)))
                errs.append(d)
            return np.sqrt(np.mean(np.square(errs)))

        errs_af.append(match_err(th_af_sorted))
        errs_bf.append(match_err(th_bf))
        errs_cs.append(match_err(th_cs))

    return (np.nanmean(errs_af), np.nanmean(errs_bf), np.nanmean(errs_cs))

In [ ]:
snr_range = np.arange(-10, 16, 3)
rmse_af, rmse_bf, rmse_cs = [], [], []
for snr in snr_range:
    a, b, c = rmse_at_snr(snr, n_trials=25)
    rmse_af.append(a); rmse_bf.append(b); rmse_cs.append(c)

In [ ]:
plt.figure(figsize=(6, 4))
plt.plot(snr_range, rmse_af, 'o-', label='Annihilating Filter')
plt.plot(snr_range, rmse_bf, 's-', label='Beamforming')
plt.plot(snr_range, rmse_cs, '^-', label='FISTA (CS)')
plt.xlabel('SNR (dB)'); plt.ylabel('RMSE (deg)')
plt.title('Monte-Carlo DoA RMSE vs SNR (2 emitters)')
plt.legend(); plt.grid(alpha=.3); plt.tight_layout()
plt.show()

print("\nDone. All 7 figures rendered inline above.")

## Part 2 — Core Algorithm Study: CMA and MUSICThe three pipelines above (AF, Beamforming, FISTA) reproduce Sharma & Suryavanshi,*"Using Small SDRs to Rapidly Validate Complex Pipelines: A Case Study in RadioDirection Finding"* (AIT Pune) — the baseline paper for this project's firstsimulation-stage validation.That paper explicitly stops short of two things it references but does not implement:- **CMA (Constant Modulus Algorithm)** — used in Kumar et al. (its own reference [4])  as the *restoration* stage of a three-step multipath-mitigation pipeline. Needed  because the paper's real-data test showed persistent, consistent false peaks caused  by room reflections (Table I / Fig. 8-10) that none of AF/Beamforming/FISTA could  remove, since none of them separate direct-path energy from multipath energy first.- **MUSIC / ESPRIT / MVDR** — mentioned only as what *array*-based systems use  (refs [1]-[3] in the paper), explicitly avoided in favor of the single rotated  antenna approach. They are studied here as **core algorithms in their own right**  (classical array signal processing), not yet as a drop-in replacement for this  single-antenna setup — see the caveat in the MUSIC section below.This section implements both from scratch on synthetic data, independent of theantenna-rotation dictionary used above, so each algorithm can be understood andunit-tested on its own before combining it with the DoA pipeline.

### D. Constant Modulus Algorithm (CMA) — Multipath Restoration**What problem this solves (per Kumar et al., the paper referenced by the baselinepaper for multipath mitigation):** a constant-modulus signal (e.g. QPSK, |s[n]| = 1)picked up through a multipath channel becomes a sum of delayed, scaled, phase-rotatedcopies of itself. The sum is **not** constant modulus anymore — that's the symptomCMA is designed to undo. CMA is a *blind* (no training sequence) adaptive equalizer:it drives an FIR filter's output back toward constant modulus by gradient descent onthe Godard/CM cost function$$J(w) = \mathbb{E}\big[(|y[n]|^2 - R_2)^2\big], \qquad R_2 = \frac{\mathbb{E}[|s[n]|^4]}{\mathbb{E}[|s[n]|^2]}$$For unit-modulus QPSK, $R_2 = 1$. The stochastic-gradient (LMS-style) update is$$e[n] = y[n]\,(|y[n]|^2 - R_2), \qquad w \leftarrow w - \mu\, e^*[n]\, \mathbf{x}[n]$$This is symbol-rate here (not the IQ-sample/pulse-shaped rate) to keep the multipathchannel model simple and directly comparable to the signal model in Eq. 1 of bothpapers: a direct path plus one delayed, attenuated, phase-rotated reflection.

In [ ]:
# =========================================================================
# D. CONSTANT MODULUS ALGORITHM (CMA) -- multipath restoration stage
#    (Kumar et al., Sec. III-A "Restoration"; motivated by the false peaks
#     seen in the baseline paper's real-data test, Sec. IV-B)
# =========================================================================
CMA_SEED = 7
cma_rng = np.random.default_rng(CMA_SEED)

def qpsk_symbols(n, rng):
    """Unit-modulus QPSK symbols: |s[n]| = 1 exactly."""
    bits = rng.integers(0, 2, size=(n, 2))
    sym = (2 * bits[:, 0] - 1) + 1j * (2 * bits[:, 1] - 1)
    return sym / np.sqrt(2)


def multipath_channel(s, D=3, rho2=0.6, phase=0.9):
    """Two-path channel: direct path + one delayed/attenuated/rotated reflection.
    Mirrors x_p(t) = rho_1 g(.) s(t) + rho_2 g(.) s(t-tau) + noise (Eq. 1, both papers),
    with the antenna-gain factors folded into rho for this single-path-pair test."""
    s_delayed = np.concatenate([np.zeros(D, dtype=complex), s])[:len(s)]
    return s + rho2 * np.exp(1j * phase) * s_delayed


def add_noise(x, snr_db, rng):
    p = np.mean(np.abs(x) ** 2)
    noise_power = p / (10 ** (snr_db / 10))
    noise = (rng.normal(0, 1, len(x)) + 1j * rng.normal(0, 1, len(x))) * np.sqrt(noise_power / 2)
    return x + noise


def cma_equalize(x, n_taps=15, mu=1e-3, n_iter=8, R2=1.0):
    """Blind CM equalizer. Returns the equalized sequence, final tap weights,
    and the per-epoch CM cost so convergence can be inspected."""
    w = np.zeros(n_taps, dtype=complex)
    w[n_taps // 2] = 1.0  # center-spike init (standard CMA practice)
    y = np.zeros(len(x), dtype=complex)
    cost_hist = []
    for _ in range(n_iter):
        for n in range(n_taps, len(x)):
            xn = x[n - n_taps:n][::-1]
            yn = np.vdot(w, xn)              # w^H x
            e = yn * (np.abs(yn) ** 2 - R2)  # CM error
            w = w - mu * np.conj(e) * xn     # gradient step
            y[n] = yn
        cost_hist.append(np.mean((np.abs(y[n_taps:]) ** 2 - R2) ** 2))
    return y, w, np.array(cost_hist)


# --- run the demo ---
N_SYM = 3000
s_true = qpsk_symbols(N_SYM, cma_rng)
x_multipath = multipath_channel(s_true, D=3, rho2=0.6, phase=0.9)
x_multipath = add_noise(x_multipath, snr_db=25, rng=cma_rng)

y_restored, w_cma, cma_cost = cma_equalize(x_multipath, n_taps=15, mu=1e-3, n_iter=8, R2=1.0)

seg = slice(1000, 2800)  # settled region, past filter warm-up/transient
print("=== CMA restoration ===")
print(f"std |x| before CMA (multipath-corrupted): {np.std(np.abs(x_multipath[seg])):.4f}   (0 = perfectly constant modulus)")
print(f"std |y| after  CMA (restored):             {np.std(np.abs(y_restored[seg])):.4f}")
print(f"CM cost per epoch: {np.round(cma_cost, 4)}")

In [ ]:
# ---- CMA diagnostic plots: constellation before/after, and cost convergence ----
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

axes[0].scatter(x_multipath[seg].real, x_multipath[seg].imag, s=4, alpha=.3)
axes[0].set_title("Constellation BEFORE CMA\n(multipath-corrupted)")
axes[0].set_xlabel("I"); axes[0].set_ylabel("Q"); axes[0].axis("equal"); axes[0].grid(alpha=.3)

axes[1].scatter(y_restored[seg].real, y_restored[seg].imag, s=4, alpha=.3, color="C1")
axes[1].set_title("Constellation AFTER CMA\n(restored)")
axes[1].set_xlabel("I"); axes[1].set_ylabel("Q"); axes[1].axis("equal"); axes[1].grid(alpha=.3)

axes[2].plot(cma_cost, "o-")
axes[2].set_title("CMA cost function vs. epoch")
axes[2].set_xlabel("Epoch"); axes[2].set_ylabel(r"$E[(|y|^2-R_2)^2]$"); axes[2].grid(alpha=.3)

plt.tight_layout(); plt.show()

print("\nInterpretation: BEFORE, the constellation is smeared into a ring/blob because")
print("the two-path sum no longer has constant modulus. AFTER CMA converges, the four")
print("QPSK points should be recognizably separating out again -- this is the signal")
print("the Kumar et al. pipeline then uses as the reference for the SEPARATION stage")
print("(estimating per-path delay/DoA), which is the next piece to implement once this")
print("restoration stage is validated.")

**Status / what's NOT done yet:** this cell validates only the *Restoration* stageof the Kumar et al. three-step pipeline (CMA). The *Separation* stage (estimatingper-path delay $\tilde\tau_l$ via the Vandermonde/complex-frequency-estimationapproach, Eqs. 4-6 of that paper) and the *Estimation* stage (per-path DoA from theseparated power profiles) still need to be built on top of this before it plugs intothe angle-sweep pipeline above. Flagging this explicitly so it isn't reported as"done" prematurely.

### E. MUSIC (MUltiple SIgnal Classification)**Important scope note before reading this section:** MUSIC is a subspace methodthat needs a *spatial covariance matrix* built from multiple simultaneousarray-element measurements (or multiple independent snapshots across time from afixed array). The single rotated-antenna setup used in the baseline paper (and above)produces one scalar voltage reading *per angle*, not multiple simultaneous channeloutputs -- so MUSIC cannot be applied to that data directly without an adaptation(e.g. treating time-multiplexed rotation positions as a synthetic aperture, whichrequires phase coherence across the whole rotation that a hand-rotated mechanicalsetup doesn't currently guarantee).This section therefore implements **classical MUSIC on a synthetic uniform lineararray (ULA)** -- the textbook form referenced in the baseline paper's introduction([1]-[3]) -- purely to build a correct, tested implementation and understand itsmechanics (covariance estimation, eigendecomposition, noise-subspace projection).Adapting it to the single-rotated-antenna case is a separate, open task, not solvedhere.

In [ ]:
# =========================================================================
# E. MUSIC -- classical array-based subspace DoA estimation (synthetic ULA)
#    Reference framework only (paper refs [1]-[3]); NOT yet adapted to the
#    single rotated-antenna measurement model used in Sections A-D.
# =========================================================================
MUSIC_SEED = 11
music_rng = np.random.default_rng(MUSIC_SEED)

def music_ula(M=8, K=2, thetas_deg=(20.0, -30.0), n_snap=200, snr_db=15, d=0.5, rng=None):
    """Classical narrowband MUSIC for a uniform linear array.
    M       : number of array elements
    K       : number of sources (assumed known, same assumption as the AF/FISTA methods above)
    d       : element spacing in wavelengths (0.5 = standard half-wavelength ULA)
    n_snap  : number of independent time snapshots used to estimate the covariance
    """
    thetas = np.deg2rad(np.array(thetas_deg))
    # Array manifold: A[:,k] = steering vector for source k
    A = np.exp(-1j * 2 * np.pi * d * np.outer(np.arange(M), np.sin(thetas)))

    # K uncorrelated unit-power source signals over n_snap snapshots
    S = (rng.normal(0, 1, (K, n_snap)) + 1j * rng.normal(0, 1, (K, n_snap))) / np.sqrt(2)
    X = A @ S

    sig_power = np.mean(np.abs(X) ** 2)
    noise_power = sig_power / (10 ** (snr_db / 10))
    N = (rng.normal(0, 1, X.shape) + 1j * rng.normal(0, 1, X.shape)) * np.sqrt(noise_power / 2)
    X = X + N

    # Sample covariance matrix and its eigendecomposition
    Rxx = (X @ X.conj().T) / n_snap
    eigval, eigvec = np.linalg.eigh(Rxx)          # ascending order
    order = np.argsort(eigval)[::-1]               # descending
    eigvec = eigvec[:, order]
    En = eigvec[:, K:]                              # noise subspace (M-K dims)

    # MUSIC pseudo-spectrum: 1 / || a(theta)^H * En ||^2
    scan_deg = np.linspace(-90, 90, 721)
    P = np.zeros(len(scan_deg))
    for i, th in enumerate(scan_deg):
        a = np.exp(-1j * 2 * np.pi * d * np.arange(M) * np.sin(np.deg2rad(th)))
        P[i] = 1.0 / np.abs(a.conj() @ En @ En.conj().T @ a)
    return scan_deg, P, eigval[::-1]


THETAS_TRUE_MUSIC = (20.0, -30.0)
scan_deg, P_music, eigvals_sorted = music_ula(
    M=8, K=2, thetas_deg=THETAS_TRUE_MUSIC, n_snap=200, snr_db=15, rng=music_rng
)
theta_hat_music = pick_peaks(scan_deg, P_music, k=2, min_sep_deg=5)

print("=== MUSIC (synthetic ULA) ===")
print("True angles:     ", THETAS_TRUE_MUSIC)
print("Estimated angles:", theta_hat_music)
print("Eigenvalues (descending, signal/noise gap should be visible after index K=2):")
print(np.round(eigvals_sorted, 3))

In [ ]:
# ---- MUSIC diagnostic plots: pseudo-spectrum and eigenvalue spread ----
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(scan_deg, 10 * np.log10(P_music / P_music.max()))
for t in THETAS_TRUE_MUSIC:
    axes[0].axvline(t, color="k", linestyle="--", alpha=.6)
for t in theta_hat_music:
    axes[0].axvline(t, color="r", linestyle=":", alpha=.8)
axes[0].set_title("MUSIC pseudo-spectrum (synthetic ULA)")
axes[0].set_xlabel("Angle (deg)"); axes[0].set_ylabel("Normalized power (dB)")
axes[0].grid(alpha=.3)

axes[1].stem(np.arange(1, len(eigvals_sorted) + 1), eigvals_sorted, basefmt=" ")
axes[1].axvline(len(THETAS_TRUE_MUSIC) + 0.5, color="r", linestyle="--", alpha=.6,
                 label="signal/noise subspace split")
axes[1].set_title("Covariance eigenvalue spectrum")
axes[1].set_xlabel("Index"); axes[1].set_ylabel("Eigenvalue"); axes[1].legend(); axes[1].grid(alpha=.3)

plt.tight_layout(); plt.show()

### F. Summary — where each algorithm stands right now| Algorithm | Status | Data used | Directly usable on the single-rotated-antenna rig? ||---|---|---|---|| Annihilating Filter (AF) | Implemented, matches baseline paper Sec. III-A | Synthetic angular voltage profile V(θ) | Yes — this is what it was built for || Beamforming | Implemented, matches baseline paper Sec. III-B | Synthetic angular voltage profile V(θ) | Yes || FISTA / Compressive Sensing | Implemented, matches baseline paper Sec. III-C | Synthetic angular voltage profile V(θ) | Yes || **CMA** | Restoration stage only implemented and validated (this notebook) | Synthetic symbol-rate IQ, 2-path channel | **Not yet** — needs Separation + Estimation stages (Kumar et al. Secs. III-B/C) before it plugs into the angle sweep || **MUSIC** | Classical ULA form implemented and validated (this notebook) | Synthetic array snapshots (unrelated data model) | **No** — needs an adaptation (e.g. synthetic-aperture / phase-coherent rotation) that hasn't been designed yet |**Honest status for the next update to Sir:** AF, Beamforming, and FISTA are validatedagainst the baseline paper's simulation results. CMA and MUSIC are now implemented andindividually validated as *standalone algorithms* on their own synthetic test data —this is genuinely what was reported ("testing core basic algorithms ... to thoroughlyunderstand the implementation first"). What is **not** true yet, and shouldn't beimplied, is that CMA or MUSIC are integrated into the single-antenna DoA pipelineabove; that integration is the next concrete step, and for MUSIC specifically itrequires deciding how to get spatial/temporal diversity out of one rotated antennabefore the algorithm even applies.